# Logging

PyLabRobot uses standard Python [`logging`](https://docs.python.org/3/library/logging.html). All loggers live under the `"pylabrobot"` name, so the usual Python techniques for setting levels, adding handlers, and filtering by module all work exactly as you'd expect.

## Log levels

Python's five built-in levels apply as normal:

| Level | Value | Defined by | Typical content |
|-------|-------|------------|-----------------|
| `CRITICAL` | 50 | Python | Unrecoverable errors |
| `ERROR` | 40 | Python | Failures that stop an operation |
| `WARNING` | 30 | Python | Recoverable problems (timeouts, fallbacks) |
| `INFO` | 20 | Python | High-level actions: setup, teardown, measurement starts |
| `DEBUG` | 10 | Python | Internal state, retry logic, per-well progress |
| `IO` | 5 | PyLabRobot | Raw bytes sent to / received from hardware |

PyLabRobot adds one extra level: **`IO`** (value 5), which sits below `DEBUG`. It exists because hardware backends can generate a large volume of raw data frames sent to and received from hardware. Mixing those into `DEBUG` would make debug logs unreadable during normal development, so they are separated into their own level that you only enable when you are actively inspecting wire traffic executed by PyLabRobot.

This is a device-agnostic feature of PyLabRobot: regardless of whether you are talking to a Hamilton STAR over USB, a plate reader over serial, or a temperature controller over TCP, all low-level communication is logged through the same `IO` level. This means you get a consistent way to inspect and record wire traffic across every backend, using the same `verbose()` call and the same log format. When something goes wrong at the hardware level, you don't need device-specific debugging tools - just set the log level to `IO` and every byte sent and received will appear in your log.

## Quick start

In Jupyter notebooks, PLR automatically enables console logging at `INFO` level. In scripts you need to opt in.

In [ ]:
import pylabrobot

# Enable console output at INFO level (the default)
pylabrobot.verbose(True)

To see `DEBUG` messages:

In [ ]:
import logging

pylabrobot.verbose(True, level=logging.DEBUG)

To see everything including raw IO bytes:

In [ ]:
from pylabrobot.io import LOG_LEVEL_IO

pylabrobot.verbose(True, level=LOG_LEVEL_IO)

To turn console output off again:

In [ ]:
pylabrobot.verbose(False)

## What the output looks like

At **INFO** level you see high-level actions:

```
2026-03-04 17:07:58,102 - pylabrobot - INFO - Setting up the plate reader.
2026-03-04 17:07:59,003 - pylabrobot - INFO - Plate reader set up successfully.
2026-03-04 17:07:59,210 - pylabrobot - INFO - Starting fluorescence measurement.
```

At **DEBUG** you also see internal details:

```
2026-03-04 17:07:59,814 - pylabrobot - DEBUG - FL measurement progress: 5/96
2026-03-04 17:07:59,920 - pylabrobot - DEBUG - Status flags: busy=True, running=True, plate_detected=True
2026-03-04 17:08:00,015 - pylabrobot - DEBUG - Retry 1/3: waiting for device ready
```

At **IO** you additionally see every raw frame on the wire:

```
2026-03-04 17:07:59,211 - pylabrobot - IO - sent 47 bytes: 020027...0d
2026-03-04 17:07:59,318 - pylabrobot - IO - read 12 bytes: 02000a...0d
```

Each level includes all messages from the levels above it, so `IO` gives you everything.

## Logging to a file

Use `setup_logger` to write logs to disk. This is useful for long-running protocols where you want a persistent record.

In [ ]:
pylabrobot.setup_logger(log_dir="./logs", level=logging.DEBUG)

This creates a file like `logs/pylabrobot-20260312.log` (dated by day). The file handler captures everything at or above the level you set. You can combine this with `verbose()` to also see output in the console.

Pass `log_dir=None` to disable file logging.

## Config file

Instead of calling `setup_logger` in every script, you can create a `pylabrobot.ini` (or `pylabrobot.json`) in your project directory:

```ini
[logging]
level = DEBUG
log_dir = ./logs
```

```json
{
  "logging": {
    "level": "DEBUG",
    "log_dir": "./logs"
  }
}
```

PLR searches for this file in the current directory and all parent directories. The config is loaded automatically on `import pylabrobot`.

## Filtering by module

Since PLR uses Python's logger hierarchy, you can adjust the level for a specific backend without changing the global level:

In [ ]:
import logging

# Only show warnings and above from the STAR backend
logging.getLogger("pylabrobot.liquid_handling.backends.hamilton.STAR").setLevel(logging.WARNING)

# Show debug output from the plate reader
logging.getLogger("pylabrobot.plate_reading").setLevel(logging.DEBUG)

## Real-world example: protocol logging

In practice you often want PLR's hardware logs alongside your own protocol-level messages (e.g. which protocol is running, who started it, and why). You can combine `setup_logger` for PLR with standard Python loggers for your own code, all writing to the same file:


In [ ]:
import logging
from pylabrobot.io import LOG_LEVEL_IO

# 1. Let PLR handle its own file logging
pylabrobot.setup_logger(log_dir="./logs", level=LOG_LEVEL_IO)

# 2. Create your own loggers for protocol-level events
logger_protocol = logging.getLogger("protocol")
logger_protocol.setLevel(logging.INFO)

# 3. Attach them to the same file handler PLR created
plr_file_handler = next(
    h for h in logging.getLogger("pylabrobot").handlers
    if isinstance(h, logging.FileHandler)
)
logger_protocol.addHandler(plr_file_handler)

# 4. Log protocol metadata
logger_protocol.info("START AUTOMATED PROTOCOL")
logger_protocol.info("Protocol: cell_viability_assay")
logger_protocol.info("User: alice")

This gives you a single log file with both PLR hardware traces and your protocol events, without having to manually set up file handlers and formatters.

## Summary

| Goal | Code |
|------|------|
| Console output (INFO) | `pylabrobot.verbose(True)` |
| Console output (DEBUG) | `pylabrobot.verbose(True, level=logging.DEBUG)` |
| Console output (IO) | `pylabrobot.verbose(True, level=LOG_LEVEL_IO)` |
| Log to file | `pylabrobot.setup_logger(log_dir="./logs", level=logging.DEBUG)` |
| Config file | `pylabrobot.ini` or `pylabrobot.json` in project root |
| Filter one module | `logging.getLogger("pylabrobot.module").setLevel(...)` |
| Disable console | `pylabrobot.verbose(False)` |